In [0]:
%sql
--- Spending tier by Country

WITH customer_spending AS (
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        c.Country,
        SUM(il.UnitPrice * il.Quantity) AS total_spending
    FROM silver_invoiceline il
    JOIN silver_invoice i ON il.InvoiceId = i.InvoiceId
    JOIN silver_customer c ON i.CustomerId = c.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName, c.Country
),
percentile_thresholds AS (
    SELECT
        PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY total_spending) AS high_threshold,
        PERCENTILE_CONT(0.60) WITHIN GROUP (ORDER BY total_spending) AS medium_threshold
    FROM customer_spending
),
customer_tiers AS (
    SELECT
        cs.CustomerId,
        cs.FirstName,
        cs.LastName,
        cs.Country,
        cs.total_spending,
        CASE
            WHEN cs.total_spending >= pt.high_threshold THEN 'High'
            WHEN cs.total_spending >= pt.medium_threshold THEN 'Medium'
            ELSE 'Low'
        END AS spending_tier
    FROM customer_spending cs
    CROSS JOIN percentile_thresholds pt
)
SELECT
    Country,
    spending_tier,
    COUNT(*) AS customer_count,
    ROUND(SUM(total_spending), 2) AS total_revenue,
    ROUND(AVG(total_spending), 2) AS avg_spending,
    ROUND(MIN(total_spending), 2) AS min_spending,
    ROUND(MAX(total_spending), 2) AS max_spending
FROM customer_tiers
WHERE spending_tier = 'High'
GROUP BY Country, spending_tier
ORDER BY
    total_revenue DESC,
    Country,
    CASE spending_tier
        WHEN 'High' THEN 1
        WHEN 'Medium' THEN 2
        WHEN 'Low' THEN 3
    END


